# Day 35 — fine-tune the retrieval encoder and the cross-encoder (Colab/Kaggle)

The build sandbox has no torch and no hub access, so it trained the same loss
(`MultipleNegativesRankingLoss`, in-batch negatives plus mined hard negatives) over a
**frozen lexical feature matrix with a 128-dim projection** and measured it with the
same evaluation entry point this notebook uses. That run is real and reported
(`ml/reports/day35-retrieval.md`, MLflow `b4e61549403a441e8e8fd80186849f99`): dev MRR
0.5274 → 0.5642, and on the frozen test split the projection is the one setting that
clearly helps the thin Hindi and Bengali slices.

What this notebook adds is the **multilingual checkpoint** the sandbox could not train:

1. the same pairs and the same mined hard negatives, feeding a real
   `paraphrase-multilingual-MiniLM-L12-v2` fine-tune;
2. a real cross-encoder (`mmarco-mMiniLMv2-L12-H384-v1`) fitted on the *same* 1709
   labelled candidates the lexical scorer was fitted on, so the two are comparable;
3. the eight-setting evaluation again, with the checkpoint as the dense half.

Rules that travel with this file: no notebook number is quoted anywhere without an
MLflow run id; **weights are never committed** (the checkpoint stays in
`ml/artifacts/`, gitignored); and if the encoder changes, the fusion weight and the
abstain floors **must be re-fitted** — `tuning.json` records the embedder kind it was
fitted for and the backend refuses numbers from another one.

Free tier is enough: a T4 makes the encoder fine-tune a few minutes, CPU works and is
slow. Nothing here needs a GPU the sandbox lacked.


In [ ]:
# 1. Install the repository and the training stack.
!git clone --depth 1 https://github.com/BuildbyArindam/Manovia.git
%cd Manovia/ml
!pip install -q -e '.[ml,dev,training]'
!pip install -q sentence-transformers
# The backend is needed for the harness (it measures the *served* retrievers).
!pip install -q -e '../backend'


In [ ]:
# 2. Rebuild the eval set and its four splits, then check the fingerprints.
#
# If these four hashes do not match the committed ones, the data on this
# runtime is not the data the sandbox report was written against, and every
# number below is about a different eval set. Stop and find out why.
!make data

import json, pathlib
manifest = json.loads(pathlib.Path('data/retrieval_splits/manifest.json').read_text())
expected = {
    'train':       'c186b664c8f1371648e28160c46ba6061eb796e05587665d43e4ba676f2feb34',
    'dev':         'b9dc4cbbf98b10e931ad770e2f6c90ff3d6a1311cefc0630b66065a7c437173e',
    'calibration': '1ce6ecf9c07afeebc8e98e1a593625b8d7c25fa53aefaa24448ad072475e099c',
    'test':        '091a469c94b47c6ba8f8437f6d3ba11617f94128900ed2488749aee7df2f44bd',
}
for name, info in manifest['splits'].items():
    got = info['fingerprint']
    print(f"{name:12s} rows={info['rows']:4d} {'MATCH' if got == expected[name] else 'MISMATCH'}")
    assert got == expected[name], f'{name} does not match the committed split'


In [ ]:
# 3. Build the pairs and mine the hard negatives with the tested module.
#
# Mining needs *scores* over the corpus, and the scores must come from the
# encoder being replaced (an off-the-shelf checkpoint), not from the one being
# trained: negatives mined with the trained encoder move during training and the
# run stops being reproducible.
from manovia_ml.config import load_retrieval_config
from manovia_ml.paths import resolve
from manovia_ml.retrieval.corpus import build_passages, load_documents
from manovia_ml.retrieval.data import load_eval_set, split_rows
from manovia_ml.retrieval.embeddings import SentenceTransformerEncoder
from manovia_ml.retrieval.pairs import build_pairs

BASE = 'paraphrase-multilingual-MiniLM-L12-v2'
config = load_retrieval_config(resolve('configs/retrieval_finetune.yaml'))
rows = load_eval_set(resolve(config.data.eval_set))
splits = split_rows(rows, seed=config.data.split_seed)
passages = build_passages(load_documents(resolve(config.data.corpus_dir) if config.data.corpus_dir.strip() else None))
by_chunk = {passage.chunk_id: passage for passage in passages}

off_the_shelf = SentenceTransformerEncoder(BASE, batch_size=32)
pairs, mining = build_pairs(
    splits['train'],
    passages,
    encoder=off_the_shelf,
    negatives=config.training.hard_negatives,
    candidates=config.training.mine_candidates,
    max_positives_per_query=config.training.max_positives_per_query,
)
print(f'{mining.queries} queries -> {mining.pairs} pairs')
print(f'positives/query {mining.positives_per_query}  negatives/pair {mining.negatives_per_pair}')
print(f'tag-preferring negatives {mining.tag_preferring_choices}, pairs with none {mining.zero_negative_pairs}')
# The sandbox mined 147 pairs / 588 negatives (232 tag-preferring, 0 empty).
# A very different count here means a different corpus or a different encoder.


In [ ]:
# 4. The dev scorer the training loop calls: same metrics module the sandbox used.
#
# Define this cell BEFORE running cell 4 (or run it once first): the loop
# references it by name.
import time
from manovia_ml.retrieval.metrics import QueryResult, evaluate_ranking

def evaluate_dev(model, dev_rows, passages):
    """Document-level max-passage ranking of one split, scored by the tested metrics."""
    texts = [passage.embed_text for passage in passages]
    vectors = model.encode(texts, batch_size=32, convert_to_numpy=True,
                           normalize_embeddings=True)
    results = []
    for row in dev_rows:
        started = time.perf_counter()
        query = model.encode([row.query], normalize_embeddings=True)[0]
        scores = vectors @ query
        order = sorted(range(len(passages)), key=lambda i: (-float(scores[i]), passages[i].chunk_id))
        seen, ranked = set(), []
        for index in order:
            doc = passages[index].doc_id
            if doc not in seen:
                seen.add(doc)
                ranked.append(doc)
        results.append(QueryResult(
            query_id=row.id, lang=row.lang, style=row.style,
            relevant=tuple(row.relevant_docs), retrieved=tuple(ranked[:5]),
            top_score=float(scores[order[0]]),
            latency_ms=(time.perf_counter() - started) * 1000.0,
        ))
    return evaluate_ranking(results, setting='checkpoint-dev', ks=(1, 3, 5))

# Off-the-shelf reference point, measured exactly the same way. The sandbox
# baseline over lexical features was dev MRR 0.5274 / recall@1 0.2312; a
# multilingual checkpoint should beat that before any fine-tuning.
reference = SentenceTransformer(BASE)
baseline = evaluate_dev(reference, splits['dev'], passages)
print('off-the-shelf dev mrr', round(baseline.mrr, 4), 'recall@1', round(baseline.recall[1], 4))


In [ ]:
# 5. Fine-tune the checkpoint with MultipleNegativesRankingLoss.
#
# Same loss, same epochs/batch/lr, same selection rule as
# `manovia_ml.retrieval.contrastive.train_projection`: pick the epoch on dev
# MRR, never on train loss. `InputExample(texts=[query, positive, neg...])` is
# how sentence-transformers passes hard negatives to MNRL.
import torch
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

examples = [
    InputExample(texts=[pair.query, by_chunk[pair.positive].embed_text,
                        *(by_chunk[neg].embed_text for neg in pair.hard_negatives)])
    for pair in pairs
]
loader = DataLoader(examples, shuffle=True, batch_size=config.training.batch_size)
model = SentenceTransformer(BASE)
loss = losses.MultipleNegativesRankingLoss(model)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

ARTIFACT = resolve('artifacts/retrieval_encoder_v1')   # gitignored, never committed
best = {'epoch': 0, 'mrr': -1.0}
for epoch in range(1, config.training.epochs + 1):
    model.fit(train_objectives=[(loader, loss)], epochs=1, warmup_steps=0,
              output_path=None, device=device, show_progress_bar=True)
    report = evaluate_dev(model, splits['dev'], passages)   # defined in the next cell
    print(f'epoch {epoch:2d}  dev mrr {report.mrr:.4f}  recall@1 {report.recall[1]:.4f}')
    if report.mrr > best['mrr']:
        best = {'epoch': epoch, 'mrr': report.mrr}
        model.save(str(ARTIFACT))
print('best epoch', best['epoch'], 'dev mrr', round(best['mrr'], 4))
# The sandbox selection rule, for comparison: best_epoch 9 on dev MRR,
# dev MRR 0.5274 -> 0.5642, recall@1 0.2312 -> 0.2742.


In [ ]:
# 6. Re-fit the fusion weight and the abstain floors for the NEW embedder.
#
# Not optional. `tuning.json` records `embedder_kind` and the backend logs
# `rag_tuning_embedder_mismatch` and ignores numbers fitted for another
# embedder. Write a run-local config whose encoder is the checkpoint, then run
# the same two stages the sandbox ran - dev for the weight, calibration for the
# floors, test never.
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path('configs/retrieval_hybrid.yaml').read_text())
cfg['encoder'] = {'kind': 'sentence_transformers', 'model_id': str(ARTIFACT), 'dim': 384}
cfg['tracking']['tags']['encoder'] = 'checkpoint'
pathlib.Path('/tmp/retrieval_hybrid_checkpoint.yaml').write_text(yaml.safe_dump(cfg))

!python -m manovia_ml.retrieval.tuning --config /tmp/retrieval_hybrid_checkpoint.yaml \
    --stage fusion --workdir data/retrieval_artifacts/harness_checkpoint --require-backend
!python -m manovia_ml.retrieval.reranker --config /tmp/retrieval_hybrid_checkpoint.yaml \
    --workdir data/retrieval_artifacts/harness_checkpoint --require-backend
!python -m manovia_ml.retrieval.tuning --config /tmp/retrieval_hybrid_checkpoint.yaml \
    --stage abstain --workdir data/retrieval_artifacts/harness_checkpoint --require-backend


In [ ]:
# 7. A real cross-encoder, fitted on the same candidates the lexical one used.
#
# `make ml-retrieval-rerank` wrote `examples.jsonl`: every candidate the first
# stage produced for the train split, with the 12 backend features and the label.
# Fit the transformer on those rows and score it on dev, so the comparison with
# the distilled lexical scorer (dev roc_auc 0.6814 -> 0.7488 in the sandbox) is
# on identical data rather than on a different pool.
from sentence_transformers import CrossEncoder

examples_path = resolve('data/retrieval_artifacts/reranker_v1/examples.jsonl')
rows = [json.loads(line) for line in examples_path.read_text().splitlines() if line.strip()]
train_rows = [row for row in rows if row['split'] == 'train'] if 'split' in rows[0] else rows
cross = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', num_labels=1, max_length=256)
cross.fit(
    train_dataloader=DataLoader(
        [InputExample(texts=[row['query'], row['title'] + ' ' + row['text']], label=float(row['label']))
         for row in train_rows],
        shuffle=True, batch_size=16),
    epochs=2,
)
print('fitted on', len(train_rows), 'candidates;',
      sum(row['label'] for row in train_rows), 'relevant')
# Scoring it on dev, and distilling it back into the 12 lexical features, are
# the two follow-ups: `load_cross_encoder` in the backend is written to accept a
# transformer behind the same seam (ADR 0035 §4).


In [ ]:
# 8. The same eight-setting evaluation, with the checkpoint as the dense half.
#
# This is the side-by-side with `ml/reports/day35-retrieval.md`. Same split
# (test, 52 hand-written rows), same metrics, same latency protocol - the only
# difference is the encoder, which is the point.
eval_cfg = yaml.safe_load(pathlib.Path('configs/retrieval_eval.yaml').read_text())
eval_cfg['encoder'] = {'kind': 'sentence_transformers', 'model_id': str(ARTIFACT), 'dim': 384}
pathlib.Path('/tmp/retrieval_eval_checkpoint.yaml').write_text(yaml.safe_dump(eval_cfg))

!python -m manovia_ml.evaluation.retrieval --config /tmp/retrieval_eval_checkpoint.yaml \
    --workdir data/retrieval_artifacts/harness_checkpoint --require-backend

# Sandbox numbers to compare against (test split, lexical encoders):
#   baseline              mrr=0.639 r@1=0.292 r@3=0.500 r@5=0.592 p95=3.1ms
#   dense_finetuned       mrr=0.629 r@1=0.317 r@3=0.433 r@5=0.487 p95=2.8ms
#   hybrid                mrr=0.622 r@1=0.296 r@3=0.483 r@5=0.517 p95=3.3ms
#   hybrid_rerank         mrr=0.658 r@1=0.354 r@3=0.467 r@5=0.529 p95=8.5ms
#   hybrid_finetuned_rerank mrr=0.707 r@1=0.338 r@3=0.500 r@5=0.571 p95=8.3ms


## 9. What to take back to the repository

Commit **numbers and configs**, never the checkpoint:

* the generated `ml/reports/day35-retrieval.md` from cell 8, with its MLflow run id;
* the re-fitted `backend/app/ml_artifacts/retrieval_v1/tuning.json` from cell 6 — it
  carries `embedder_kind: sentence_transformers`, the new fusion weight, the new
  floors and the four split fingerprints, which is the evidence that the test split
  was still not used to tune anything;
* `docs/model-card.md` §5.5 and `docs/verification/day35-retrieval.md` §1 updated with
  the checkpoint rows, keeping the offline-projection rows beside them (they are the
  ablation, and deleting them would hide what the sandbox actually measured);
* `PROGRESS.md`: what the checkpoint changed, and whether the recommendation moved.

Two things to check before believing any of it:

1. **The split fingerprints in cell 2 matched.** If they did not, the run measured a
   different eval set and the comparison with the sandbox table is meaningless.
2. **`RAG_EMBEDDER=sentence_transformers` and `RAG_EMBEDDING_MODEL_ID` point at the
   checkpoint directory** in the deployment, and the ingest was re-run
   (`make ingest`): the store keeps the vectors it was built with, and a fine-tuned
   encoder over a store embedded by another one is a silent mismatch. The embedder
   marker (`embedder.json`, `EMBEDDER_FIT_VERSION`) exists to make that loud — check
   the `rag_retriever_built` log line names the embedder you think it does.

Serving the transformer cross-encoder from cell 7 is deliberately **not** done here:
it needs an export behind `load_cross_encoder` (ONNX, the Day-34 path) and a
re-calibrated floor, because a threshold fitted on lexical scores means nothing on
transformer scores. That is parked in `PROGRESS.md`.
